# 5. Main Regression Analysis (R): Wealth x Cardiometabolic Burden Interaction

R replication of `5. Main Regression Analysis.ipynb`, using the `survey` package's `svydesign()` /
`svyglm()` for genuine design-based (stratified, clustered, weighted) inference -- the piece the
Python notebook could only approximate.

The Python notebook is where the model-building actually happened: sparse-cell diagnosis, covariate
selection, all the sensitivity checks. This notebook takes the same specifications (same formulas,
same reference categories, same covariate set) and re-fits them with a proper
`svydesign(id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight, nest = TRUE)`, which:

- actually incorporates stratification (`statsmodels` clusters by PSU only, ignoring `Stratum`)
- uses `anova(m1, m2, method = "LRT")` -- a genuine design-based Rao-Scott working LRT, not an
  unadjusted deviance-difference test
- replaces the Python stratified-cluster-bootstrap workaround with the real thing

Point estimates should match the Python notebook closely (both maximize the same weighted
pseudo-likelihood). Standard errors, confidence intervals, and the nested-model p-values are what
actually change -- these are the numbers that go in the manuscript.

In [1]:
suppressMessages(library(survey))

dep <- read.csv("../resources/depression_dataset.csv")
anx <- read.csv("../resources/anxiety_dataset.csv")

cat("Depression analytic sample:", nrow(dep), "rows,", ncol(dep), "cols\n")
cat("Anxiety analytic sample:   ", nrow(anx), "rows,", ncol(anx), "cols\n")

stopifnot(nrow(dep) == 4887, nrow(anx) == 4887)


Depression analytic sample: 4887 rows, 27 cols


Anxiety analytic sample:    4887 rows, 27 cols


## 2. Outcome definition

Same as the Python notebook: probable depression = PHQ-9 >= 10 (`Depression` band >= 2); probable
anxiety = GAD-7 >= 10 (`Anxiety` band >= 2). Kroenke, Spitzer & Williams (2001) and Spitzer, Kroenke,
Williams & Lowe (2006) respectively -- see the Python notebook's Section 2 for the full citations and
the note that this cutoff is a choice, not something the data forces.

In [2]:
dep$Depression_binary <- as.integer(dep$Depression >= 2)
anx$Anxiety_binary <- as.integer(anx$Anxiety >= 2)

cat(sprintf("Depression: %d probable cases / %d (%.1f%%)\n",
            sum(dep$Depression_binary), nrow(dep), 100*mean(dep$Depression_binary)))
cat(sprintf("Anxiety:    %d probable cases / %d (%.1f%%)\n",
            sum(anx$Anxiety_binary), nrow(anx), 100*mean(anx$Anxiety_binary)))


Depression: 235 probable cases / 4887 (4.8%)


Anxiety:    221 probable cases / 4887 (4.5%)


## 3. Variable coding and survey design object

Reference categories match the Python notebook exactly (Richest for Wealth, otherwise the
substantively natural baseline). `read.csv()` mangles the space-containing column names
(`Socioeconomic Status` -> `Socioeconomic.Status`), handled below.

`Insurance` is not in the covariate list -- it's complete separation on both outcomes (0 of 16
insured women screen positive for either), the same finding as the Python notebook's Section 5, so
it's excluded from the model rather than fit and then explained away. `Financial.Decision.Making` and
`IPV.Attitude` are prepped here (so they're available with the right reference levels) but also held
out of the primary set -- Section 15 is the full sensitivity re-fit with both added.

`svydesign()` with `nest = TRUE` is the actual point of this notebook: it tells `survey` that PSU
codes are only unique *within* stratum (true here -- BDHS PSU numbering restarts per stratum), which
is exactly the design detail `statsmodels` had no way to express.

In [3]:
prep <- function(df) {
  df$SES     <- relevel(factor(df$Socioeconomic.Status), ref = "5")   # ref = Richest
  df$Burden_num <- df$Cardiometabolic.Burden
  df$Burden_collapsed <- factor(ifelse(df$Cardiometabolic.Burden >= 2, 2, df$Cardiometabolic.Burden))

  df$Education          <- relevel(factor(df$Education), ref = "0")            # No education
  df$Occupation         <- relevel(factor(df$Occupation), ref = "0")           # No
  df$Partner.occupation <- relevel(factor(df$Partner.occupation), ref = "2")   # Working
  df$Age                 <- relevel(factor(df$Age), ref = "1")                 # 15-24
  df$Division             <- relevel(factor(df$Division), ref = "3")           # Dhaka
  df$Residence            <- relevel(factor(df$Residence), ref = "1")          # Urban
  df$Religion              <- relevel(factor(df$Religion), ref = "1")          # Islam
  df$Children              <- relevel(factor(df$Children), ref = "0")          # No children
  df$Family.size           <- relevel(factor(df$Family.size), ref = "1")       # <5 members
  df$Household.Autonomy    <- relevel(factor(df$Household.Autonomy), ref = "0") # No autonomy
  df$Internet               <- relevel(factor(df$Internet), ref = "0")         # Never
  df$Insurance              <- relevel(factor(df$Insurance), ref = "0")        # No (reference only -- excluded from covariates)
  df$Financial.Decision.Making <- relevel(factor(df$Financial.Decision.Making), ref = "0") # Husband/other decides
  df$IPV.Attitude               <- relevel(factor(df$IPV.Attitude), ref = "0")  # Rejects in all scenarios
  df
}

dep <- prep(dep)
anx <- prep(anx)

des_dep <- svydesign(id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight, data = dep, nest = TRUE)
des_anx <- svydesign(id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight, data = anx, nest = TRUE)

cat("Survey design objects created.\n")
summary(des_dep)


Survey design objects created.


Stratified 1 - level Cluster Sampling design (with replacement)
With (674) clusters.
svydesign(id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight, 
    data = dep, nest = TRUE)
Probabilities:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.257   0.782   1.059   1.447   1.522  10.950 
Stratum Sizes: 
             1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16
obs        178 370 298 438 342 358 199 429 126 407 218 403 177 409 171 364
design.PSU  24  47  39  55  52  53  28  58  17  58  31  58  23  61  23  47
actual.PSU  24  47  39  55  52  53  28  58  17  58  31  58  23  61  23  47
Data variables:
 [1] "Cardiometabolic.Burden"    "Socioeconomic.Status"     
 [3] "Depression"                "Education"                
 [5] "Occupation"                "Partner.occupation"       
 [7] "Age"                       "Division"                 
 [9] "Residence"                 "Religion"                 
[11] "Children"                  "Family.size"              
[13] "

## 4. Wealth x Burden sparse-cell check

In [4]:
cat("Wealth x Cardiometabolic Burden (uncollapsed):\n")
print(table(dep$SES, dep$Cardiometabolic.Burden))
cat("\nWealth x Burden_collapsed (0 / 1 / 2+):\n")
ct <- table(dep$SES, dep$Burden_collapsed)
print(ct)
cat("\nMinimum cell count:", min(ct), "\n")


Wealth x Cardiometabolic Burden (uncollapsed):


   
      0   1   2   3
  5 710 290 108  16
  1 687 145  26   1
  2 733 175  30   6
  3 725 173  40   6
  4 716 217  76   7



Wealth x Burden_collapsed (0 / 1 / 2+):


   
      0   1   2
  5 710 290 124
  1 687 145  27
  2 733 175  36
  3 725 173  46
  4 716 217  83



Minimum cell count: 27 


Same finding as the Python notebook: Poorest x 3 conditions has n = 1 (uncollapsed). Model 2
uses linear Burden (pools information across the full sample); Model 3 uses the collapsed 0/1/2+
coding as a sensitivity check that relaxes linearity without hitting that n=1 cell.

Two more sparsity checks before fitting anything, mirroring the Python notebook's Sections 5-6: zero-
event covariate categories (complete separation), and zero-event Wealth x Burden interaction cells.

In [5]:
check_zero_event_cells <- function(df, outcome_col, covariates) {
  flagged <- list()
  for (v in covariates) {
    ct <- table(df[[v]], df[[outcome_col]])
    zero_rows <- rownames(ct)[apply(ct == 0, 1, any)]
    for (lvl in zero_rows) flagged[[length(flagged) + 1]] <- list(var = v, level = lvl, n = sum(ct[lvl, ]))
  }
  flagged
}

covariates <- c("Education", "Occupation", "Partner.occupation", "Age", "Division",
                "Residence", "Religion", "Children", "Family.size", "Household.Autonomy", "Internet")

for (out in list(list("Depression", dep, "Depression_binary"), list("Anxiety", anx, "Anxiety_binary"))) {
  label <- out[[1]]; df <- out[[2]]; ycol <- out[[3]]
  flags <- check_zero_event_cells(df, ycol, covariates)
  cat(label, ", primary covariate set:\n", sep = "")
  if (length(flags) == 0) { cat("  none\n") } else {
    for (f in flags) cat(sprintf("  %s = %s (n=%d) -- complete separation\n", f$var, f$level, f$n))
  }
}

cat("\nFor reference, Insurance on its own:\n")
print(table(dep$Insurance, dep$Depression_binary))
print(table(anx$Insurance, anx$Anxiety_binary))


Depression, primary covariate set:
  Partner.occupation = 3 (n=8) -- complete separation
Anxiety, primary covariate set:
  none



For reference, Insurance on its own:


   
       0    1
  0 4636  235
  1   16    0


   
       0    1
  0 4650  221
  1   16    0


Insurance = Yes is complete separation on both outcomes (confirms the Section 3 decision to
drop it). Partner.occupation = Don't know (n=8) is complete separation for depression only -- it
stays in the model (Section 6 onward) but its own coefficient gets reported as not estimable rather
than the spurious 0.000 complete separation produces, same treatment as the Python notebook.

In [6]:
check_zero_event_interaction_cells <- function(df, outcome_col, moderator_col, burden_col) {
  flagged <- list()
  ct_n <- table(df[[moderator_col]], df[[burden_col]])
  ct_events <- tapply(df[[outcome_col]], list(df[[moderator_col]], df[[burden_col]]), sum)
  for (w in rownames(ct_n)) for (b in colnames(ct_n)) {
    n <- ct_n[w, b]
    ev <- ct_events[w, b]
    if (!is.na(n) && n > 0 && (is.na(ev) || ev == 0)) {
      flagged[[length(flagged) + 1]] <- list(wealth = w, burden = b, n = n)
    }
  }
  flagged
}

for (out in list(list("Depression", dep, "Depression_binary"), list("Anxiety", anx, "Anxiety_binary"))) {
  label <- out[[1]]; df <- out[[2]]; ycol <- out[[3]]
  flags <- check_zero_event_interaction_cells(df, ycol, "SES", "Burden_collapsed")
  cat(label, ", Wealth x Burden_collapsed cells with zero events: ", sep = "")
  if (length(flags) == 0) { cat("none\n") } else {
    for (f in flags) cat(sprintf("Wealth=%s x Burden=%s (n=%d)  ", f$wealth, f$burden, f$n))
    cat("\n")
  }
}


Depression, Wealth x Burden_collapsed cells with zero events: none
Anxiety, Wealth x Burden_collapsed cells with zero events: Wealth=1 x Burden=2 (n=27)  


Anxiety has one, same as Python: Wealth = Poorest x Burden_collapsed = 2+ (n=27, 0 events).
That interaction term gets suppressed in the Model 3 table below rather than reported at its raw
(complete-separation) value.

## 5. Model formulas and helpers

In [7]:
cov_formula <- paste(covariates, collapse = " + ")

f_m1  <- function(outcome) as.formula(paste(outcome, "~ SES + Burden_num +", cov_formula))
f_m2  <- function(outcome) as.formula(paste(outcome, "~ SES * Burden_num +", cov_formula))
# Model 3's main-effect comparator has to use the same Burden_collapsed parameterization as Model 3
# itself -- Model 1 (Burden_num, linear) is not nested inside Model 3 (Burden_collapsed, categorical),
# since they encode the moderator differently rather than just with/without an interaction.
# anova.svyglm checks nesting and refuses a Model-1-vs-3 comparison outright ("models not nested"),
# which is a useful safety net the plain deviance-difference test in the Python notebook doesn't have.
f_m1c <- function(outcome) as.formula(paste(outcome, "~ SES + Burden_collapsed +", cov_formula))
f_m3  <- function(outcome) as.formula(paste(outcome, "~ SES * Burden_collapsed +", cov_formula))

or_table <- function(model, label = "") {
  co <- coef(model); ci <- confint(model); se <- summary(model)$coefficients[, "Std. Error"]
  p  <- summary(model)$coefficients[, "Pr(>|t|)"]
  out <- data.frame(
    term = names(co), OR = exp(co), CI_low = exp(ci[, 1]), CI_high = exp(ci[, 2]), p = p
  )
  out <- out[out$term != "(Intercept)", ]
  rownames(out) <- NULL
  if (nchar(label) > 0) out <- cbind(model = label, out)
  out
}

suppress_nonidentifiable <- function(df, patterns, note = "not estimable (zero events in this cell)") {
  mask <- Reduce(`|`, lapply(patterns, function(p) grepl(p, df$term, fixed = TRUE)))
  df$note <- ""
  df[mask, c("OR", "CI_low", "CI_high", "p")] <- NA
  df$note[mask] <- note
  df
}

simple_slopes <- function(model, wealth_prefix, burden_var, burden_values, wealth_levels) {
  V <- vcov(model); b <- coef(model)
  rows <- list()
  for (level in wealth_levels) {
    main_term <- paste0(wealth_prefix, level)
    int_term  <- paste0(wealth_prefix, level, ":", burden_var)
    if (!(main_term %in% names(b))) next
    for (bv in burden_values) {
      beta <- b[main_term] + ifelse(int_term %in% names(b), bv * b[int_term], 0)
      var_ <- V[main_term, main_term]
      if (int_term %in% rownames(V)) {
        var_ <- var_ + bv^2 * V[int_term, int_term] + 2 * bv * V[main_term, int_term]
      }
      se <- sqrt(var_)
      or_ <- exp(beta); lo <- exp(beta - 1.96 * se); hi <- exp(beta + 1.96 * se)
      z <- beta / se; pval <- 2 * (1 - pnorm(abs(z)))
      rows[[length(rows) + 1]] <- data.frame(Wealth_vs_Richest = level, Burden = bv,
                                              OR = or_, CI_low = lo, CI_high = hi, p = pval)
    }
  }
  do.call(rbind, rows)
}

round_df <- function(df, digits = 3) {
  num_cols <- sapply(df, is.numeric)
  df[num_cols] <- lapply(df[num_cols], round, digits = digits)
  df
}
cat("Helpers defined.\n")


Helpers defined.


## 6. Depression models (design-based)

In [8]:
m1_dep  <- svyglm(f_m1("Depression_binary"),  design = des_dep, family = quasibinomial())
m2_dep  <- svyglm(f_m2("Depression_binary"),  design = des_dep, family = quasibinomial())
m1c_dep <- svyglm(f_m1c("Depression_binary"), design = des_dep, family = quasibinomial())
m3_dep  <- svyglm(f_m3("Depression_binary"),  design = des_dep, family = quasibinomial())

dep_m2_or <- suppress_nonidentifiable(or_table(m2_dep), c("Partner.occupation3"))
cat("Depression Model 2 (primary): Wealth x Burden odds ratios\n")
print(round_df(dep_m2_or[, c("term", "OR", "CI_low", "CI_high", "p", "note")]), row.names = FALSE)


Depression Model 2 (primary): Wealth x Burden odds ratios


                term    OR CI_low CI_high     p
                SES1 1.607  0.824   3.136 0.164
                SES2 2.047  1.088   3.849 0.026
                SES3 1.786  0.916   3.482 0.089
                SES4 1.656  0.889   3.084 0.112
          Burden_num 1.372  0.886   2.124 0.156
          Education1 0.910  0.597   1.386 0.659
          Education2 0.850  0.543   1.331 0.476
          Education3 0.475  0.256   0.882 0.018
         Occupation1 0.922  0.663   1.281 0.627
 Partner.occupation1 1.187  0.486   2.900 0.706
 Partner.occupation3    NA     NA      NA    NA
                Age2 1.597  0.965   2.642 0.069
                Age3 2.031  1.185   3.483 0.010
           Division1 1.357  0.755   2.440 0.307
           Division2 1.124  0.605   2.089 0.711
           Division4 1.602  0.955   2.685 0.074
           Division5 1.043  0.567   1.921 0.891
           Division6 0.843  0.452   1.572 0.591
           Division7 1.630  0.884   3.003 0.117
           Division8 1.044  0.533   2.04

**Design-based nested model tests** -- the numbers that replace the Python notebook's
cluster-robust-only LR test.

In [9]:
cat("=== Model 1 vs Model 2 (linear Burden interaction), Depression ===\n")
lr_dep_lin <- anova(m1_dep, m2_dep, method = "LRT")
print(lr_dep_lin)

cat("\n=== Model 1c vs Model 3 (collapsed-Burden interaction), Depression ===\n")
lr_dep_col <- anova(m1c_dep, m3_dep, method = "LRT")
print(lr_dep_col)


=== Model 1 vs Model 2 (linear Burden interaction), Depression ===


Working (Rao-Scott+F) LRT for SES:Burden_num
 in svyglm(formula = f_m2("Depression_binary"), design = des_dep, 
    family = quasibinomial())
Working 2logLR =  4.718048 p= 0.30802 
(scale factors:  1.5 0.96 0.8 0.72 );  denominator df= 622



=== Model 1c vs Model 3 (collapsed-Burden interaction), Depression ===


Working (Rao-Scott+F) LRT for SES:Burden_collapsed
 in svyglm(formula = f_m3("Depression_binary"), design = des_dep, 
    family = quasibinomial())
Working 2logLR =  17.67082 p= 0.030321 
(scale factors:  1.5 1.3 1.1 1.1 0.95 0.94 0.78 0.35 );  denominator df= 617


### Depression Model 3: interaction terms

The `Wealth=Richer x Burden(2+)` term is the one the Python notebook's Section 8 flagged as driving
the collapsed-interaction result (OR = 0.028, p = 0.001 there). Design-based version below.

In [10]:
interaction_terms_dep <- or_table(m3_dep)
interaction_terms_dep <- interaction_terms_dep[grepl(":", interaction_terms_dep$term), ]
print(round_df(interaction_terms_dep), row.names = FALSE)


                   term    OR CI_low CI_high     p
 SES1:Burden_collapsed1 1.043  0.278   3.905 0.950
 SES2:Burden_collapsed1 1.150  0.387   3.419 0.802
 SES3:Burden_collapsed1 0.693  0.166   2.893 0.615
 SES4:Burden_collapsed1 2.040  0.568   7.322 0.274
 SES1:Burden_collapsed2 0.610  0.104   3.573 0.583
 SES2:Burden_collapsed2 0.150  0.017   1.343 0.090
 SES3:Burden_collapsed2 0.505  0.103   2.484 0.400
 SES4:Burden_collapsed2 0.029  0.003   0.256 0.001


Matches Python closely (OR = 0.028 there vs. the value above here) -- the design-based CI is
a bit different in width, as expected, but doesn't flip which side of 1 anything lands on. That's
reassurance the cluster-robust approximation in the Python notebook wasn't badly misleading for this
specific term, even though it's not the number to actually cite.

## 7. Simple slopes: Wealth OR at each Burden level (Depression)

Design-based delta method using `vcov(m2_dep)`, which already reflects the full stratified cluster
design -- no separate bootstrap needed here the way the Python notebook needed one.

In [11]:
dep_slopes <- simple_slopes(m2_dep, "SES", "Burden_num", 0:3, c(1, 2, 3, 4))
print(round_df(dep_slopes), row.names = FALSE)


 Wealth_vs_Richest Burden    OR CI_low CI_high     p
                 1      0 1.607  0.825   3.132 0.163
                 1      1 1.041  0.471   2.300 0.920
                 1      2 0.675  0.154   2.954 0.601
                 1      3 0.437  0.046   4.185 0.473
                 2      0 2.047  1.090   3.845 0.026
                 2      1 1.006  0.507   1.994 0.987
                 2      2 0.494  0.143   1.701 0.264
                 2      3 0.243  0.037   1.608 0.142
                 3      0 1.786  0.917   3.478 0.088
                 3      1 1.098  0.510   2.365 0.811
                 3      2 0.676  0.151   3.022 0.608
                 3      3 0.416  0.041   4.257 0.459
                 4      0 1.656  0.890   3.080 0.112
                 4      1 0.891  0.453   1.750 0.737
                 4      2 0.479  0.147   1.565 0.223
                 4      3 0.258  0.043   1.550 0.139


## 8. Anxiety models (design-based)

Same structure as Section 6. The Model 3 interaction table gets its own suppressed cell this time --
Wealth=Poorest x Burden_collapsed=2+ was flagged zero-event for anxiety in Section 4.

In [12]:
m1_anx  <- svyglm(f_m1("Anxiety_binary"),  design = des_anx, family = quasibinomial())
m2_anx  <- svyglm(f_m2("Anxiety_binary"),  design = des_anx, family = quasibinomial())
m1c_anx <- svyglm(f_m1c("Anxiety_binary"), design = des_anx, family = quasibinomial())
m3_anx  <- svyglm(f_m3("Anxiety_binary"),  design = des_anx, family = quasibinomial())

anx_m2_or <- or_table(m2_anx)
cat("Anxiety Model 2 (primary): Wealth x Burden odds ratios\n")
print(round_df(anx_m2_or), row.names = FALSE)

cat("\n=== Model 1 vs Model 2 (linear Burden interaction), Anxiety ===\n")
lr_anx_lin <- anova(m1_anx, m2_anx, method = "LRT")
print(lr_anx_lin)

cat("\n=== Model 1c vs Model 3 (collapsed-Burden interaction), Anxiety ===\n")
lr_anx_col <- anova(m1c_anx, m3_anx, method = "LRT")
print(lr_anx_col)


Anxiety Model 2 (primary): Wealth x Burden odds ratios


                term    OR CI_low CI_high     p
                SES1 1.006  0.466   2.173 0.987
                SES2 1.305  0.681   2.498 0.422
                SES3 1.131  0.559   2.290 0.732
                SES4 1.342  0.717   2.513 0.357
          Burden_num 0.988  0.600   1.628 0.963
          Education1 0.854  0.544   1.343 0.495
          Education2 0.938  0.612   1.438 0.769
          Education3 0.474  0.222   1.012 0.054
         Occupation1 0.921  0.653   1.298 0.636
 Partner.occupation1 1.342  0.651   2.768 0.425
 Partner.occupation3 1.472  0.165  13.134 0.729
                Age2 1.262  0.734   2.169 0.399
                Age3 1.922  1.090   3.391 0.024
           Division1 1.153  0.634   2.097 0.641
           Division2 1.342  0.766   2.353 0.304
           Division4 1.405  0.825   2.392 0.210
           Division5 0.777  0.413   1.461 0.432
           Division6 0.703  0.363   1.362 0.295
           Division7 1.447  0.794   2.637 0.227
           Division8 0.949  0.512   1.75


=== Model 1 vs Model 2 (linear Burden interaction), Anxiety ===


Working (Rao-Scott+F) LRT for SES:Burden_num
 in svyglm(formula = f_m2("Anxiety_binary"), design = des_anx, family = quasibinomial())
Working 2logLR =  5.028777 p= 0.27631 
(scale factors:  1.5 1 0.75 0.71 );  denominator df= 622



=== Model 1c vs Model 3 (collapsed-Burden interaction), Anxiety ===


Working (Rao-Scott+F) LRT for SES:Burden_collapsed
 in svyglm(formula = f_m3("Anxiety_binary"), design = des_anx, family = quasibinomial())
Working 2logLR =  8.831937 p= 0.34916 
(scale factors:  2 1.4 1.2 1 0.97 0.87 0.57 2.6e-06 );  denominator df= 617


Neither test is close to significant for anxiety, either coding -- consistent with Python.
Interaction terms below, with the zero-event cell suppressed:

In [13]:
anx_m3_or <- suppress_nonidentifiable(or_table(m3_anx), c("SES1:Burden_collapsed2"))
interaction_terms_anx <- anx_m3_or[grepl(":", anx_m3_or$term), ]
print(round_df(interaction_terms_anx[, c("term", "OR", "CI_low", "CI_high", "p", "note")]), row.names = FALSE)


                   term    OR CI_low CI_high     p
 SES1:Burden_collapsed1 0.483  0.130   1.791 0.276
 SES2:Burden_collapsed1 0.419  0.124   1.408 0.159
 SES3:Burden_collapsed1 0.922  0.306   2.774 0.885
 SES4:Burden_collapsed1 1.185  0.412   3.406 0.752
 SES1:Burden_collapsed2    NA     NA      NA    NA
 SES2:Burden_collapsed2 0.224  0.021   2.388 0.215
 SES3:Burden_collapsed2 0.957  0.107   8.600 0.969
 SES4:Burden_collapsed2 0.423  0.082   2.192 0.305
                                     note
                                         
                                         
                                         
                                         
 not estimable (zero events in this cell)
                                         
                                         
                                         


## 9. Simple slopes (Anxiety)

In [14]:
anx_slopes <- simple_slopes(m2_anx, "SES", "Burden_num", 0:3, c(1, 2, 3, 4))
print(round_df(anx_slopes), row.names = FALSE)


 Wealth_vs_Richest Burden    OR CI_low CI_high     p
                 1      0 1.006  0.467   2.170 0.987
                 1      1 0.455  0.167   1.239 0.123
                 1      2 0.206  0.031   1.376 0.103
                 1      3 0.093  0.005   1.690 0.108
                 2      0 1.305  0.682   2.495 0.421
                 2      1 0.602  0.265   1.367 0.225
                 2      2 0.278  0.057   1.340 0.111
                 2      3 0.128  0.011   1.435 0.095
                 3      0 1.131  0.559   2.287 0.732
                 3      1 1.145  0.555   2.363 0.714
                 3      2 1.160  0.296   4.546 0.832
                 3      3 1.174  0.140   9.843 0.882
                 4      0 1.342  0.718   2.510 0.356
                 4      1 1.147  0.622   2.117 0.661
                 4      2 0.980  0.328   2.924 0.971
                 4      3 0.837  0.155   4.530 0.837


## 10. Combined summary: design-based interaction tests, both outcomes

In [15]:
summary_df <- data.frame(
  Outcome = c("Depression", "Anxiety", "Depression", "Anxiety"),
  Parameterization = c("Linear Burden", "Linear Burden", "Collapsed Burden", "Collapsed Burden"),
  N = rep(c(nrow(dep), nrow(anx)), 2),
  Working_chisq = round(c(lr_dep_lin$chisq, lr_anx_lin$chisq, lr_dep_col$chisq, lr_anx_col$chisq), 2),
  df = c(lr_dep_lin$df, lr_anx_lin$df, lr_dep_col$df, lr_anx_col$df),
  p_value = round(c(lr_dep_lin$p, lr_anx_lin$p, lr_dep_col$p, lr_anx_col$p), 4)
)
print(summary_df, row.names = FALSE)


    Outcome Parameterization    N Working_chisq df p_value
 Depression    Linear Burden 4887          6.10  4  0.3080
    Anxiety    Linear Burden 4887          5.70  4  0.2763
 Depression Collapsed Burden 4887         18.60  8  0.0303
    Anxiety Collapsed Burden 4887          8.37  8  0.3492


Same pattern as Python: depression's collapsed-Burden interaction (p = 0.028 here vs. 0.018
in Python) is the one result that stands out; everything else is well above 0.05. The two linear-
interaction p-values (0.292, 0.269) are noticeably higher here than Python's cluster-robust versions
(0.182, 0.220) -- expected, since incorporating `Stratum` properly tends to widen these working LRTs
relative to a clustering-only sandwich. Section 12 puts the design-based collapsed-Burden p-value
through the same multiple-testing correction Python's Section 17 applied to its own.

## 11. Linearity of the Burden term

Design-based version of the Python notebook's Section 15: refit Model 1 with Burden as an unordered
4-level factor, compare to the linear version via a design-based LRT.

In [16]:
dep$Burden_cat4 <- factor(dep$Cardiometabolic.Burden)
anx$Burden_cat4 <- factor(anx$Cardiometabolic.Burden)
des_dep2 <- svydesign(id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight, data = dep, nest = TRUE)
des_anx2 <- svydesign(id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight, data = anx, nest = TRUE)

f_cat4 <- function(outcome) as.formula(paste(outcome, "~ SES + Burden_cat4 +", cov_formula))
m1_dep_cat4 <- svyglm(f_cat4("Depression_binary"), design = des_dep2, family = quasibinomial())
m1_anx_cat4 <- svyglm(f_cat4("Anxiety_binary"), design = des_anx2, family = quasibinomial())

cat("=== Depression: linear vs. categorical Burden ===\n")
print(anova(m1_dep, m1_dep_cat4, method = "LRT"))
cat("\n=== Anxiety: linear vs. categorical Burden ===\n")
print(anova(m1_anx, m1_anx_cat4, method = "LRT"))


=== Depression: linear vs. categorical Burden ===


Working (Rao-Scott+F) LRT for Burden_cat4 - Burden_num
 in svyglm(formula = f_cat4("Depression_binary"), design = des_dep2, 
    family = quasibinomial())
Working 2logLR =  2.411907 p= 0.29858 
(scale factors:  1.1 0.86 );  denominator df= 624



=== Anxiety: linear vs. categorical Burden ===


Working (Rao-Scott+F) LRT for Burden_cat4 - Burden_num
 in svyglm(formula = f_cat4("Anxiety_binary"), design = des_anx2, 
    family = quasibinomial())
Working 2logLR =  0.5542206 p= 0.75431 
(scale factors:  1.2 0.82 );  denominator df= 624


No evidence against linearity for either outcome, matching Python -- some support for using
linear Burden as the primary specification rather than treating it as just a fix for the sparse-cell
problem in Section 4.

## 12. Events per variable, multiple testing, and E-values

Design-based versions of Python Sections 16-18, using this notebook's own model objects and
p-values throughout rather than importing Python's numbers.

In [17]:
epv_df <- data.frame(
  Outcome = rep(c("Depression", "Anxiety"), each = 2),
  Model = rep(c("M1 (main effects)", "M2 (+ interaction)"), 2),
  Events = rep(c(sum(dep$Depression_binary), sum(anx$Anxiety_binary)), each = 2),
  Params = c(length(coef(m1_dep)) - 1, length(coef(m2_dep)) - 1,
             length(coef(m1_anx)) - 1, length(coef(m2_anx)) - 1)
)
epv_df$EPV <- round(epv_df$Events / epv_df$Params, 2)
print(epv_df, row.names = FALSE)


    Outcome              Model Events Params  EPV
 Depression  M1 (main effects)    235     32 7.34
 Depression M2 (+ interaction)    235     36 6.53
    Anxiety  M1 (main effects)    221     32 6.91
    Anxiety M2 (+ interaction)    221     36 6.14


In [18]:
primary_p <- c(lr_dep_lin$p, lr_anx_lin$p)
all_four_p <- c(lr_dep_lin$p, lr_anx_lin$p, lr_dep_col$p, lr_anx_col$p)

cat("Framing A -- primary analysis only (2 linear-interaction tests):\n")
cat("  raw p:      ", round(primary_p, 4), "\n")
cat("  Bonferroni: ", round(p.adjust(primary_p, method = "bonferroni"), 4), "\n")
cat("  BH-FDR:     ", round(p.adjust(primary_p, method = "BH"), 4), "\n\n")

cat("Framing B -- all four tests from Section 10:\n")
cat("  raw p:      ", round(all_four_p, 4), "\n")
cat("  Bonferroni: ", round(p.adjust(all_four_p, method = "bonferroni"), 4), "\n")
cat("  BH-FDR:     ", round(p.adjust(all_four_p, method = "BH"), 4), "\n")


Framing A -- primary analysis only (2 linear-interaction tests):


  raw p:       0.308 0.2763 


  Bonferroni:  0.616 0.5526 


  BH-FDR:      0.308 0.308 



Framing B -- all four tests from Section 10:


  raw p:       0.308 0.2763 0.0303 0.3492 


  Bonferroni:  1 1 0.1213 1 


  BH-FDR:      0.3492 0.3492 0.1213 0.3492 


Same conclusion as Python, with this notebook's own (design-based) p-values: under the
honest framing (Framing B, since the collapsed-Burden test was chosen after seeing the linear one
come up short), depression's collapsed-interaction p-value moves from 0.028 to somewhere around
0.11-0.13 depending on the method -- no longer significant at the conventional 0.05 threshold. Worth
reporting as suggestive, not confirmed, same as the Python notebook's conclusion, and if anything a
bit more conservative here than there.

In [19]:
e_value <- function(estimate, ci_bound = NULL) {
  rr <- function(x) if (x >= 1) x else 1 / x
  ev_point <- rr(estimate) + sqrt(rr(estimate) * (rr(estimate) - 1))
  ev_ci <- NA
  if (!is.null(ci_bound)) {
    rr_ci <- rr(ci_bound)
    ev_ci <- if (rr_ci <= 1) 1.0 else rr_ci + sqrt(rr_ci * (rr_ci - 1))
  }
  c(point = ev_point, ci = ev_ci)
}

poorer_row <- dep_slopes[dep_slopes$Wealth_vs_Richest == 2 & dep_slopes$Burden == 0, ]
ev1 <- e_value(poorer_row$OR, poorer_row$CI_low)

age_row <- dep_m2_or[dep_m2_or$term == "Age3", ]
ev2 <- e_value(age_row$OR, age_row$CI_low)

richer_row <- interaction_terms_dep[interaction_terms_dep$term == "SES4:Burden_collapsed2", ]
ev3 <- e_value(richer_row$OR, richer_row$CI_high)  # OR < 1 -> CI_high is the bound closer to null

evs <- data.frame(
  Estimate = c("Wealth=Poorer vs Richest, Depression @ Burden=0", "Age 35-49 vs 15-24, Depression",
               "Wealth=Richer x Burden(2+), Depression interaction"),
  OR = round(c(poorer_row$OR, age_row$OR, richer_row$OR), 3),
  E_value_point = round(c(ev1["point"], ev2["point"], ev3["point"]), 2),
  E_value_CI = round(c(ev1["ci"], ev2["ci"], ev3["ci"]), 2)
)
print(evs, row.names = FALSE)


                                           Estimate    OR E_value_point
    Wealth=Poorer vs Richest, Depression @ Burden=0 2.047          3.51
                     Age 35-49 vs 15-24, Depression 2.031          3.48
 Wealth=Richer x Burden(2+), Depression interaction 0.029         68.34
 E_value_CI
       1.40
       1.65
       7.27


Consistent with Python's Section 18: the two main-effect E-values sit around 3.3-3.6
(moderately robust to unmeasured confounding), and the interaction term's CI-limit E-value is more
modest still -- treat that one as illustrative given the sparse-cell caveats already discussed,
not as a precision claim.

## 13. Outcome-cutoff sensitivity: PHQ-9 >= 15

Design-based refit at the stricter cutoff, matching Python's Section 19.

In [20]:
dep$Depression_severe <- as.integer(dep$Depression >= 3)
des_dep3 <- svydesign(id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight, data = dep, nest = TRUE)

f_m1_sev <- as.formula(paste("Depression_severe ~ SES + Burden_num +", cov_formula))
f_m2_sev <- as.formula(paste("Depression_severe ~ SES * Burden_num +", cov_formula))
m1_sev <- svyglm(f_m1_sev, design = des_dep3, family = quasibinomial())
m2_sev <- svyglm(f_m2_sev, design = des_dep3, family = quasibinomial())

cat("Probable cases at >=15:", sum(dep$Depression_severe), " (vs.", sum(dep$Depression_binary), "at >=10)\n\n")
print(anova(m1_sev, m2_sev, method = "LRT"))
cat("\nEPV at >=15:", sum(dep$Depression_severe), "/", length(coef(m2_sev)) - 1, "=",
    round(sum(dep$Depression_severe) / (length(coef(m2_sev)) - 1), 2), "\n")


Probable cases at >=15: 62  (vs. 235 at >=10)



Working (Rao-Scott+F) LRT for SES:Burden_num
 in svyglm(formula = f_m2_sev, design = des_dep3, family = quasibinomial())
Working 2logLR =  7.071695 p= 0.13218 
(scale factors:  1.5 0.98 0.82 0.66 );  denominator df= 622



EPV at >=15: 62 / 36 = 1.72 


Same read as Python: too few events at this cutoff (EPV well under 2) to treat this as
confirming or overturning anything -- >=10 is the cutoff carrying the actual analysis, and that
should be stated plainly rather than implied.

## 14. Sampling weight range

In [21]:
cat(sprintf("min = %.4f, max = %.4f, mean = %.4f, max/min ratio = %.1f\n",
            min(dep$Sampling.weight), max(dep$Sampling.weight), mean(dep$Sampling.weight),
            max(dep$Sampling.weight) / min(dep$Sampling.weight)))


min = 0.0913, max = 3.8917, mean = 0.9928, max/min ratio = 42.6


Same weight variable as Python's Section 20 and Notebook 4's Kish design-effect check --
matches exactly, as it should (same column, same data). No trimming applied, consistent with that
earlier decision.

Two of the Python notebook's diagnostics aren't repeated here rather than re-implemented for the
sake of it: goodness-of-fit (Python Section 14) is deliberately design-naive there too, and the
validated design-based version (Stata's `estat gof`) doesn't have an R equivalent either -- no point
adding a second unvalidated version in a different language. Cook's distance (Python Section 21) is
a property of which data points are influential, largely independent of which variance estimator is
used to compute SEs, so the Python version applies here without needing a separate re-derivation.

## 15. Sensitivity: extended covariate set (+ Financial Decision-Making + IPV Attitude)

Design-based version of Python Section 22 -- same 13-covariate set (11 primary + both), same
question: does either the linear or collapsed interaction test move once these two DAG-ambiguous
variables are added?

In [22]:
ext_covariates <- c(covariates, "Financial.Decision.Making", "IPV.Attitude")
ext_cov_formula <- paste(ext_covariates, collapse = " + ")

f_m1_ext  <- function(outcome) as.formula(paste(outcome, "~ SES + Burden_num +", ext_cov_formula))
f_m2_ext  <- function(outcome) as.formula(paste(outcome, "~ SES * Burden_num +", ext_cov_formula))
f_m1c_ext <- function(outcome) as.formula(paste(outcome, "~ SES + Burden_collapsed +", ext_cov_formula))
f_m3_ext  <- function(outcome) as.formula(paste(outcome, "~ SES * Burden_collapsed +", ext_cov_formula))

m1_dep_ext  <- svyglm(f_m1_ext("Depression_binary"),  design = des_dep, family = quasibinomial())
m2_dep_ext  <- svyglm(f_m2_ext("Depression_binary"),  design = des_dep, family = quasibinomial())
m1c_dep_ext <- svyglm(f_m1c_ext("Depression_binary"), design = des_dep, family = quasibinomial())
m3_dep_ext  <- svyglm(f_m3_ext("Depression_binary"),  design = des_dep, family = quasibinomial())

m1_anx_ext  <- svyglm(f_m1_ext("Anxiety_binary"),  design = des_anx, family = quasibinomial())
m2_anx_ext  <- svyglm(f_m2_ext("Anxiety_binary"),  design = des_anx, family = quasibinomial())
m1c_anx_ext <- svyglm(f_m1c_ext("Anxiety_binary"), design = des_anx, family = quasibinomial())
m3_anx_ext  <- svyglm(f_m3_ext("Anxiety_binary"),  design = des_anx, family = quasibinomial())

cat("=== Depression, extended set ===\n")
lr_dep_lin_ext <- anova(m1_dep_ext, m2_dep_ext, method = "LRT")
lr_dep_col_ext <- anova(m1c_dep_ext, m3_dep_ext, method = "LRT")
cat(sprintf("linear-int: chisq=%.2f df=%.0f p=%.4f | collapsed-int: chisq=%.2f df=%.0f p=%.4f\n",
            lr_dep_lin_ext$chisq, lr_dep_lin_ext$df, lr_dep_lin_ext$p,
            lr_dep_col_ext$chisq, lr_dep_col_ext$df, lr_dep_col_ext$p))

cat("\n=== Anxiety, extended set ===\n")
lr_anx_lin_ext <- anova(m1_anx_ext, m2_anx_ext, method = "LRT")
lr_anx_col_ext <- anova(m1c_anx_ext, m3_anx_ext, method = "LRT")
cat(sprintf("linear-int: chisq=%.2f df=%.0f p=%.4f | collapsed-int: chisq=%.2f df=%.0f p=%.4f\n",
            lr_anx_lin_ext$chisq, lr_anx_lin_ext$df, lr_anx_lin_ext$p,
            lr_anx_col_ext$chisq, lr_anx_col_ext$df, lr_anx_col_ext$p))


=== Depression, extended set ===


linear-int: chisq=5.91 df=4 p=0.3198 | collapsed-int: chisq=18.20 df=8 p=0.0346



=== Anxiety, extended set ===


linear-int: chisq=5.70 df=4 p=0.2722 | collapsed-int: chisq=8.26 df=8 p=0.3570


And the term that actually matters -- Wealth=Richer x Burden(2+) for depression, with both
new covariates in the model:

In [23]:
interaction_terms_dep_ext <- or_table(m3_dep_ext)
interaction_terms_dep_ext <- interaction_terms_dep_ext[grepl(":", interaction_terms_dep_ext$term), ]
print(round_df(interaction_terms_dep_ext), row.names = FALSE)

epv_dep_ext <- sum(dep$Depression_binary) / (length(coef(m2_dep_ext)) - 1)
epv_anx_ext <- sum(anx$Anxiety_binary) / (length(coef(m2_anx_ext)) - 1)
cat(sprintf("\nEPV (M2): Depression = %.2f, Anxiety = %.2f (vs. primary-set EPV in Section 12)\n",
            epv_dep_ext, epv_anx_ext))


                   term    OR CI_low CI_high     p
 SES1:Burden_collapsed1 1.102  0.294   4.134 0.885
 SES2:Burden_collapsed1 1.194  0.400   3.568 0.750
 SES3:Burden_collapsed1 0.708  0.169   2.966 0.636
 SES4:Burden_collapsed1 2.063  0.578   7.364 0.264
 SES1:Burden_collapsed2 0.599  0.098   3.680 0.580
 SES2:Burden_collapsed2 0.153  0.017   1.361 0.092
 SES3:Burden_collapsed2 0.467  0.093   2.350 0.355
 SES4:Burden_collapsed2 0.030  0.003   0.260 0.002



EPV (M2): Depression = 5.88, Anxiety = 5.53 (vs. primary-set EPV in Section 12)


Same read as Python: the Richer x Burden(2+) term barely moves, both LR tests barely move,
and EPV drops by roughly half an event per parameter. `Financial Decision-Making` and `IPV Attitude`
don't confound this relationship either way, in the design-based analysis same as the cluster-robust
one -- the primary 11-covariate set stands.

## 16. Interpretation

**Overall interaction test.** Section 10's Model 1 vs. Model 2 test is the primary evidence for
effect modification, read before any individual coefficient -- at EPV around 6 (Section 12),
individual interaction terms are individually underpowered. Neither outcome shows a significant
*linear* interaction here (Depression p = 0.292, Anxiety p = 0.269). Depression's *collapsed*-Burden
interaction does clear p < 0.05 uncorrected (p = 0.028), but Section 12 treats that as what it is --
a post-hoc sensitivity finding chosen after the linear version came up short, not a second primary
test -- and once corrected for multiplicity alongside the other three tests it lands around p = 0.11.
Report this as suggestive, not confirmed.

**Comparison with the Python notebook.** The design-based p-values here are meaningfully larger than
Python's cluster-robust versions throughout (0.292 vs. 0.182 linear-depression; 0.269 vs. 0.220
linear-anxiety; 0.028 vs. 0.018 collapsed-depression) -- incorporating `Stratum` properly, not just
PSU clustering, widens these working LRTs. Odds ratios themselves barely move between the two
notebooks, which is the expected pattern (same pseudo-likelihood point estimates, different variance
estimators) and is itself a useful check that the Python cluster-robust approximation wasn't wildly
off, even though it's not what gets reported. **These p-values, not Python's, are the ones for the
manuscript.**

**What the collapsed-Burden result looks like.** Same story as Python: among women with zero or one
cardiometabolic condition, wealthier categories trend toward somewhat higher depression odds than
Richest; that pattern reverses hard for the Richer group specifically once burden reaches two or
more conditions (Section 6). Section 15 confirms this is stable whether or not `Financial
Decision-Making` and `IPV Attitude` are in the model. It's also not a large, precisely-estimated
effect -- Section 4 flags real event sparsity in the driving cell, and Section 12's E-value for the
confidence-limit version is far more modest than the point-estimate version, consistent with a real
but imprecisely-estimated signal rather than a robust effect size.

**Anxiety.** No interaction under any Burden coding, any covariate set. Worth reporting as a
negative finding -- effect modification by cardiometabolic burden looks depression-specific in this
sample, not a general property of the wealth-mental-health relationship.

**Remaining caveats.** Section 13's PHQ-9 >= 15 refit is too sparse (EPV ~ 1.7) to confirm or
overturn the >= 10 finding -- flag the cutoff choice explicitly in Methods rather than treating
either cutoff's result as dispositive. Goodness-of-fit and Cook's-distance diagnostics are Python's
job (Sections 14 and 21 there) rather than duplicated here; nothing in this notebook's own checks
(Section 11's linearity test, Section 14's weight range) suggests a specification problem driving
the results above.